<p align="center">
<a href="https://duckietown.com"><img src="../assets/images/dtlogo.png" alt="Duckietown Logo" width="40%"></a>
</p>

<span style="float:right">
<strong>Credits:</strong>
This notebook adapts material from the <a href="https://docs.ros.org/en/jazzy/">ROS 2 documentation</a> (<a href="https://creativecommons.org/licenses/by/4.0/">CC BY 4.0</a>) and from the Duckietown <em>ROS Basics</em> and <em>ROS 2 Middleware</em> learning experiences.
</span>

# ROS 2 Subscribers and Diagnosis

The previous notebook connected the publisher's source code to the published messages, and drove the Duckiebot with a publishing node. On the receiving side, an application needs its own subscription so incoming data can trigger useful behaviour.

This notebook reviews how subscribers process each message, first with the text publisher and then with the camera and encoders of the Duckiebot, and how to diagnose common subscriber failures, such as a subscriber producing no output. Following a message into its callback, then stopping its publisher, makes the difference between a waiting subscriber and a working data path observable. The quality-of-service settings of sensor streams get a closer look.

## Part 1: a local subscriber

### Prepare the publisher

Use three terminals in the editor, with a router reachable (notebook 03): one for the publisher, one for the subscriber, and one for inspection. Stop any earlier exercise nodes. Start the publisher in Terminal 1 (with the rate changed to 2 Hz in notebook 09, or set it with `--ros-args -p rate_hz:=2.0`):

```bash
ros2 run ros2_basics my_publisher_node --ros-args -p rate_hz:=2.0
```

The publisher logs its text twice per second. Keep it running.

### Read and run the subscriber

Open [my_subscriber_node.py](../packages/ros2_basics/ros2_basics/my_subscriber_node.py) and observe its settings:

```python
NODE_NAME = "my_subscriber_node"
TOPIC_TO_SUBSCRIBE = "chatter"
QUEUE_DEPTH = 10
```

The topic and message type match the publisher. The subscriber has its own node name, so graph tools can identify the receiving application separately. Inside the constructor, the subscription is created with:

```python
self._subscription = self.create_subscription(String, TOPIC_TO_SUBSCRIBE, self._on_chatter, QUEUE_DEPTH)
```

The arguments specify the message type, topic, callback and history depth. As with the publisher's timer callback, `self._on_chatter` is passed without parentheses so the executor can invoke it later. Incoming messages provide the work for the callback:

```python
def _on_chatter(self, msg: String) -> None:
    self.get_logger().info(f'I heard: "{msg.data}"')
```

The callback receives a `String` message and reads its `data` field. While `rclpy.spin(node)` processes ready work, each invocation logs the received text. The constructor creates three more subscriptions, for the robot's encoders and camera; they stay silent in this part because no robot data arrives.

In Terminal 2:

```bash
ros2 run ros2_basics my_subscriber_node
```

After discovery, lines such as these should appear:

```shell
[INFO] [1793450010.237398723] [my_subscriber_node]: I heard: "Hello from ROS 2! [41]"
[INFO] [1793450010.739041393] [my_subscriber_node]: I heard: "Hello from ROS 2! [42]"
```

The text matches the publisher's payload, with approximately half a second between messages. `my_subscriber_node` received the messages of `my_publisher_node` and executed its callback.

### Inspect both ends of the connection

Keep both nodes running and, in Terminal 3:

```bash
ros2 node list --no-daemon --spin-time 5
```

```shell
/my_publisher_node
/my_subscriber_node
```

With both nodes visible, their endpoint details establish how they are connected:

```bash
ros2 node info /my_publisher_node
ros2 node info /my_subscriber_node
```

The first output includes `Publishers: /chatter: std_msgs/msg/String`; the second `Subscribers: /chatter: std_msgs/msg/String`. Both endpoints use `/chatter` and the same type, while the subscriber's log supplies evidence that messages actually reach its callback.

### Diagnose missing messages

Stop only the publisher with `Ctrl-C` in Terminal 1. Leave the subscriber running. After any messages already in transit have been processed, Terminal 2 becomes quiet without returning to its shell prompt. Its callback logs incoming messages, so silence is expected when none arrive.

Does the subscriber still exist? In Terminal 3:

```bash
ros2 node list --no-daemon --spin-time 5
ros2 topic info /chatter
```

```shell
/my_subscriber_node

Type: std_msgs/msg/String
Publisher count: 0
Subscription count: 1
```

The topic remains discoverable through the subscription, although no publisher currently supplies data to it. An independent receiving check should therefore also remain quiet:

```bash
ros2 topic echo --once --timeout 3 /chatter std_msgs/msg/String
```

No message appears, and the command returns after three seconds. The explicit type lets the observer create its subscription even while the publisher is absent. Restart the publisher in Terminal 1:

```bash
ros2 run ros2_basics my_publisher_node --ros-args -p rate_hz:=2.0
```

The subscriber resumes printing `I heard:` lines after discovery, without needing a restart. Its subscription remained available while it waited for new messages.

If a subscriber remains quiet while a publisher is running, the same observations guide the next check:

- a missing node calls for inspecting its process terminal and its router connection (notebook 04);
- different resolved topic names or message types call for checking the endpoint configuration (`ros2 node info`); a node started in a namespace subscribes to `/NAMESPACE/chatter`, not `/chatter` (notebook 11 deals with this);
- matching endpoints call for checking delivery policies and attempting an independent `ros2 topic echo`.

If the independent observer receives data while the application stays quiet, attention turns to the application's subscription settings, executor and callback. After any source correction, rebuild and restart the affected node.

Stop both nodes with `Ctrl-C` when the observations are complete.

## Part 2: subscribe to the Duckiebot's sensors

The constructor of `my_subscriber_node` also subscribes to the robot's streams:

```python
self.create_subscription(WheelEncoderStamped, "left_wheel_encoder_driver_node/tick",
                         self._on_left_tick, qos_profile_sensor_data)
self.create_subscription(WheelEncoderStamped, "right_wheel_encoder_driver_node/tick",
                         self._on_right_tick, qos_profile_sensor_data)
self.create_subscription(CompressedImage, "camera_node/image/compressed",
                         self._on_image, qos_profile_sensor_data)
```

Three differences with the text subscription deserve attention.

**Relative names.** `camera_node/image/compressed` has no leading `/`: in the root namespace it would resolve to `/camera_node/image/compressed`, which does not exist. The node must run in the robot's namespace, which is what its launch file does with `namespace=veh`.

**The sensor-data QoS profile.** Instead of a queue depth, the fourth argument is `rclpy.qos.qos_profile_sensor_data`: reliability `BEST_EFFORT`, history `KEEP_LAST (5)`, durability `VOLATILE`. It encodes what a subscriber to a sensor stream wants: the freshest reading, soon, rather than every reading, late. It is the profile to use for images and encoder ticks in every node of this learning experience. Two reasons:

- *compatibility*: a publisher and a subscriber only connect if their policies are compatible. A `BEST_EFFORT` subscriber matches any publisher; a `RELIABLE` subscriber does not match a `BEST_EFFORT` publisher, which is how many sensor drivers publish. `rmw_zenoh_cpp`, the middleware of the Duckiebot, is lenient about this particular rule, but the DDS middlewares used elsewhere in ROS 2 enforce it strictly, and a subscriber that never receives anything with no error message is the typical result. Writing sensor subscribers with the sensor-data profile makes the code correct everywhere;
- *freshness*: with a small, best-effort queue, a subscriber that is too slow drops old frames instead of accumulating a growing backlog of stale images.

**Cheap callbacks.** `_on_image` only counts the frame and stores the message; decoding (`compressed_to_bgr`) happens in the report timer, every two seconds. Decoding every frame at 30 Hz in the callback would delay every other callback of the node.

The `_report` timer summarises what arrived:

```python
def _report(self) -> None:
    ...
    self.get_logger().info(" | ".join(parts))
```

### Run it against the robot

With the robot's ROS 2 stack up and a terminal pointed at the robot's router (the `export ZENOH_CONFIG_OVERRIDE=...` line of notebook 08):

```bash
ros2 launch ros2_basics my_subscriber_node.launch.py veh:=ROBOTNAME
```

Every two seconds the node reports:

```shell
[INFO] [ROBOTNAME.my_subscriber_node]: left encoder: 1532 ticks (135/rev) | right encoder: 1498 ticks (135/rev) | camera: 58 frames, last 640x480 jpeg
```

The frame count increases by the camera rate times two seconds; the tick counts change when the wheels turn (drive the robot from notebook 09 or turn a wheel by hand). Note that `chatter` is now `/ROBOTNAME/chatter`: the text publisher of Part 1, started in the root namespace, is no longer heard. Launch it in the same namespace (`ros2 launch ros2_basics my_publisher_node.launch.py veh:=ROBOTNAME`) and the `I heard:` lines are back.

### Compare the delivery policies

In another terminal pointed at the robot:

```bash
ros2 topic info --verbose /ROBOTNAME/camera_node/image/compressed
```

The output now lists two endpoints: the publisher of `camera_node` (`RELIABLE`, `KEEP_LAST (1)`) and the subscription of `my_subscriber_node` (`BEST_EFFORT`, `KEEP_LAST (5)`). This is a compatible pair, and the frame count in the report proves that images flow. `ros2 topic hz /ROBOTNAME/camera_node/image/compressed` gives the rate at which they do; compare it with the increase of the frame counter.

## The same, from the notebook

The notebook session subscribes in the same way (`sensor=True` selects the sensor-data profile). The cell below collects encoder ticks for a few seconds in a Python list, the simplest form of a subscriber callback.

In [ ]:
ROBOT = "ROBOTNAME"   # <-- the name of your (real or virtual) Duckiebot

In [ ]:
import time
from duckietown_msgs.msg import WheelEncoderStamped
from lx_ros2_tools.session import RosSession

ros = RosSession(ROBOT)

ticks = []
sub = ros.subscribe("left_wheel_encoder_driver_node/tick", WheelEncoderStamped, lambda msg: ticks.append(msg.data), sensor=True)
time.sleep(5)             # the callback runs in the background while we wait
ros.unsubscribe(sub)
print(f"received {len(ticks)} tick messages in 5 s; first {ticks[:1]}, last {ticks[-1:]}")

Zero messages while the robot stands still is normal for the encoders (the driver publishes when the wheel turns); the camera never stops:

In [ ]:
from sensor_msgs.msg import CompressedImage
from lx_ros2_tools.image import compressed_to_bgr

frames = []
sub = ros.subscribe("camera_node/image/compressed", CompressedImage, lambda msg: frames.append(msg), sensor=True)
time.sleep(3)
ros.unsubscribe(sub)
img = compressed_to_bgr(frames[-1])
print(f"received {len(frames)} frames in 3 s (about {len(frames) / 3:.0f} Hz); last frame {img.shape[1]}x{img.shape[0]} pixels, {len(frames[-1].data)} bytes")

`compressed_to_bgr` turns the JPEG bytes into a NumPy array (height x width x 3), the starting point of any image processing you will do in later learning experiences.

## Checklist: a subscriber receives nothing

1. **Is the publisher running and publishing?** `ros2 node list --no-daemon --spin-time 5`, `ros2 topic hz TOPIC`.
2. **Same router?** `printenv ZENOH_CONFIG_OVERRIDE` in both terminals; `ros2 daemon stop` after changing it.
3. **Same resolved name?** `ros2 node info` on both nodes: namespaces and remappings change names (notebook 11).
4. **Same type?** `ros2 topic info` reports one type per topic; a mismatch is an error at subscription time.
5. **Compatible QoS?** `ros2 topic info --verbose`; use the sensor-data profile for sensor streams.
6. **Does an independent observer receive?** `ros2 topic echo --once --timeout 5 TOPIC TYPE` (`--no-arr` for images). If it does, the problem is in the application's subscription, executor or callback.

The subscriber's callback connected received data to application behaviour. Stopping and restarting the publisher showed why a visible subscription can remain quiet, and node inspection, endpoint details and an independent observer helped locate the missing part of a data path. The robot's sensors added the two ingredients of a real subscriber: relative names in the robot's namespace and the sensor-data QoS profile.

## Further reading

The ROS 2 Jazzy tutorial on [writing a simple Python publisher and subscriber](https://docs.ros.org/en/jazzy/Tutorials/Beginner-Client-Libraries/Writing-A-Simple-Py-Publisher-And-Subscriber.html) provides another example of subscription creation and callback execution; the [QoS documentation](https://docs.ros.org/en/jazzy/Concepts/Intermediate/About-Quality-of-Service-Settings.html) lists the profiles and the compatibility rules.

Let's move to [the next notebook about names and remapping](../notebooks/11_names_and_remapping.ipynb).